In [17]:
import pandas as pd
import json
import warnings
import numpy as np
import os
import numpy as np
import sys



warnings.filterwarnings('ignore')
RANDOM = True


warnings.filterwarnings('ignore')
url =  "../estadistica_stop/ESTADISTICA_DELITO.csv"  #"save3.to_csv("data2.csv",compression='xz', sep='\t', index=False)"
df = pd.read_csv(url)


In [18]:


# Función para verificar que el número de filas no cambie inesperadamente
def verificar_integridad(df_actual, df_anterior, nombre_paso, debe_crecer=False):
    filas_actuales = len(df_actual)
    filas_anteriores = len(df_anterior)
    
    if debe_crecer:
        if filas_actuales <= filas_anteriores:
            print(f"❌ ERROR EN {nombre_paso}: Se esperaba que las filas aumentaran, pero se mantuvieron o redujeron.")
            print(f"   Antes: {filas_anteriores} | Después: {filas_actuales}")
            sys.exit("Ejecución detenida por inconsistencia de datos.")
    else:
        if filas_actuales != filas_anteriores:
            print(f"❌ ERROR EN {nombre_paso}: El número de filas cambió inesperadamente.")
            print(f"   Antes: {filas_anteriores} | Después: {filas_actuales}")
            print(f"   Esto indica un producto cartesiano (duplicados en el merge).")
            sys.exit("Ejecución detenida por riesgo de MemoryError o datos corruptos.")
    
    print(f"✅ {nombre_paso}: {filas_actuales} líneas (OK)")

# =========================================
# 2. CALCULAR TOTALES POR COMUNA/SEMANA
# =========================================
# Asumimos que 'df' ya está cargado antes de este bloque
lineas_inicio = len(df)

# Totales
totales = (
    df.groupby(['codcom', 'id_semana'], as_index=False)['frecuencia']
      .sum()
)
totales['delito'] = 'Total'

# Dimensión tiempo
dim_tiempo = df[['id_semana', 'semana_detalle', 'fecha']].drop_duplicates()
totales = totales.merge(dim_tiempo, on='id_semana', how='left')

# Asegurar mismas columnas que df
totales = totales.reindex(columns=df.columns, fill_value=np.nan)

# Concatenar
df = pd.concat([df, totales], ignore_index=True)
verificar_integridad(df, pd.DataFrame(index=range(lineas_inicio)), "Concatenar Totales", debe_crecer=True)


# =========================================
# 3. PREPARACIÓN TEMPORAL
# =========================================
df['fecha'] = pd.to_datetime(df['fecha'], errors='coerce')
df['año'] = df['fecha'].dt.year
df['mes'] = df['fecha'].dt.month

# Semana numérica desde semana_detalle (ej: "Semana 05")
df['semana_numero'] = (
    df['semana_detalle']
    .astype(str)
    .str.extract(r'(\d{1,2})')
    .astype(float)
)

# Orden crítico
df = df.sort_values(['delito', 'codcom', 'id_semana']).reset_index(drop=True)


# =========================================
# 4. VARIABLES BASE
# =========================================
df['casos_semana_actual'] = df['frecuencia']
df['casos_semana_anterior'] = (
    df.groupby(['delito', 'codcom'])['frecuencia'].shift(1)
)
df['delta'] = df['casos_semana_actual'] - df['casos_semana_anterior']
verificar_integridad(df, df, "Variables Base", debe_crecer=False) # Debe ser igual


# =========================================
# 5. ACUMULADOS
# =========================================
df['acumulado_anual'] = (
    df.groupby(['delito', 'codcom', 'año'])['frecuencia'].cumsum()
)
df['acumulado_total'] = (
    df.groupby(['delito', 'codcom'])['frecuencia'].cumsum()
)

# Acumulado año anterior
df_prev_acum = df[['delito','codcom','año','semana_numero','acumulado_anual']].copy()

# 🔴 CORRECCIÓN CLAVE: Eliminar duplicados antes de cambiar el año para evitar MemoryError
# Si hay múltiples registros para la misma semana/comuna, el merge explotará.
# Como 'cumsum' se calculó por grupo, el valor es el mismo para todos los registros de la clave,
# así que es seguro eliminar duplicados.
df_prev_acum = df_prev_acum.drop_duplicates(subset=['delito','codcom','año','semana_numero'])

df_prev_acum['año'] += 1
df_prev_acum.rename(
    columns={'acumulado_anual':'acumulado_anual_anterior'},
    inplace=True
)

df_len_before_merge = len(df)
df = df.merge(
    df_prev_acum,
    on=['delito','codcom','año','semana_numero'],
    how='left'
)

# Verificar que el merge no multiplicó las filas
verificar_integridad(df, pd.DataFrame(index=range(df_len_before_merge)), "Merge Acumulado Año Anterior")


# =========================================
# 6. MEDIAS MÓVILES
# =========================================
df['media_movil_4s'] = (
    df.groupby(['delito','codcom'])['frecuencia']
      .transform(lambda x: x.rolling(4, min_periods=1).mean())
)

df['media_movil_8s'] = (
    df.groupby(['delito','codcom'])['frecuencia']
      .transform(lambda x: x.rolling(8, min_periods=1).mean())
)


# =========================================
# 7. HISTÓRICOS
# =========================================
df['promedio_hist'] = (
    df.groupby(['delito','codcom'])['frecuencia']
      .transform(lambda x: x.expanding().mean())
)

df['std_hist'] = (
    df.groupby(['delito','codcom'])['frecuencia']
      .transform(lambda x: x.expanding().std())
)

df['max_hist'] = (
    df.groupby(['delito','codcom'])['frecuencia']
      .transform(lambda x: x.expanding().max())
)


# =========================================
# 8. ESTADÍSTICAS AÑO ANTERIOR
# =========================================
df['promedio_hist_anual'] = (
    df.groupby(['delito','codcom','año'])['frecuencia']
      .transform(lambda x: x.expanding().mean())
)

df['std_hist_anual'] = (
    df.groupby(['delito','codcom','año'])['frecuencia']
      .transform(lambda x: x.expanding().std())
)

df['max_hist_anual'] = (
    df.groupby(['delito','codcom','año'])['frecuencia']
      .transform(lambda x: x.expanding().max())
)

stats_prev = df[['delito','codcom','año','semana_numero',
                 'promedio_hist_anual','std_hist_anual','max_hist_anual']].copy()

# 🔴 CORRECCIÓN CLAVE: Eliminar duplicados antes del merge
# Evita el MemoryError: Unable to allocate 19.6 GiB...
stats_prev = stats_prev.drop_duplicates(subset=['delito','codcom','año','semana_numero'])

stats_prev['año'] += 1
stats_prev.rename(columns={
    'promedio_hist_anual':'promedio_hist_anual_prev',
    'std_hist_anual':'std_hist_anual_prev',
    'max_hist_anual':'max_hist_anual_prev'
}, inplace=True)

df_len_before_stats = len(df)
df = df.merge(
    stats_prev,
    on=['delito','codcom','año','semana_numero'],
    how='left'
)

# Verificar integridad
verificar_integridad(df, pd.DataFrame(index=range(df_len_before_stats)), "Merge Stats Año Anterior")

df['promedio_hist_anual'] = df['promedio_hist_anual_prev'].fillna(df['promedio_hist_anual'])
df['std_hist_anual'] = df['std_hist_anual_prev'].fillna(df['std_hist_anual'])
df['max_hist_anual'] = df['max_hist_anual_prev'].fillna(df['max_hist_anual'])

df.drop(columns=[
    'promedio_hist_anual_prev',
    'std_hist_anual_prev',
    'max_hist_anual_prev'
], inplace=True)


# =====================================================
# 9. TENDENCIA Y RACHA
# =====================================================
df['tendencia_corto_plazo'] = np.where(
    df['delta'] > 0, 'Alza',
    np.where(df['delta'] < 0, 'Baja', 'Estable')
)

# Nota: La lógica de racha original podría fallar si hay NaNs en delta, 
# pero mantenemos tu lógica original.
df['racha'] = (
    (df['delta'] > 0)
    .astype(int)
    .groupby((df['delta'] <= 0).cumsum())
    .cumsum()
)

# =========================================
# 10. MÉTRICAS AVANZADAS
# =========================================
df['var_pct_vs_semana_anterior'] = (
    df['delta'] / df['casos_semana_anterior'].replace(0, np.nan) * 100
)

df['z_score'] = (
    (df['frecuencia'] - df['promedio_hist']) /
    df['std_hist'].replace(0, np.nan)
)

df['z_score_vs_año_anterior'] = (
    (df['frecuencia'] - df['promedio_hist_anual']) /
    df['std_hist_anual'].replace(0, np.nan)
)

df['conclusion_z'] = pd.cut(
    df['z_score'].fillna(0),
    bins=[-np.inf, -2, 2, np.inf],
    labels=['Bajo', 'Normal', 'Alto']
)

df['tendencia_corto_plazo'] = np.where(
    df['delta'] > 0, 'Alza',
    np.where(df['delta'] < 0, 'Baja', 'Estable')
)

# =========================================
# 11. LOCALIZACIÓN Y RANKING
# =========================================
# Carga de datos externos
try:
    localiza = pd.read_excel(r"D:\GitHub\LOCALIZA_DB\Localiza Chile (1).xlsx")
    localiza2 = (
        localiza[['Provincia', 'Comuna', 'Región', 'Codcom', 'Codreg']]
        .drop_duplicates()
    )

    df2 = df.merge(
        localiza2,
        left_on='codcom',
        right_on='Codcom',
        how='left'
    )
    
    # Verificar integridad (Left join no debe aumentar filas)
    verificar_integridad(df2, df, "Merge Localización")

    # Ranking comunal regional
    df2 = df2.sort_values(['Codreg', 'delito', 'Codcom', 'id_semana'])

    df2['ranking_comunal_regional'] = (
        df2.groupby(['Codreg', 'delito', 'id_semana'])['frecuencia']
           .rank(method='dense', ascending=False)
    )

    df2['ranking_comunal_regional_semana_anterior'] = (
        df2.groupby(['Codreg', 'delito', 'Codcom'])['ranking_comunal_regional']
           .shift(1)
    )
    
    verificar_integridad(df2, df2, "Cálculo Rankings")

except FileNotFoundError as e:
    print(f"⚠️ Advertencia: No se encontraron archivos de localización. {e}")
    df2 = df # Continuar sin localización si no hay archivo

# =========================================
# 12. FACTORES POBLACIÓN
# =========================================
try:
    clasePoblacion = pd.read_excel(
        r"C:\Users\limc_\Downloads\Factores Población.xlsx",
        sheet_name="Clase Población"
    )
    factor = pd.read_excel(
        r"C:\Users\limc_\Downloads\Factores Población.xlsx",
        sheet_name="Factores"
    )

    clasePoblacion2 = clasePoblacion[['Codcom', 'Población', 'Clase Población']].copy()
    clasePoblacion2.columns = ['Codcom', 'poblacion_clase', 'clase_poblacion']

    factor2 = factor[['Codcom', 'Año', 'Población', 'Factor Población']].copy()
    factor2.columns = ['Codcom', 'año', 'poblacion', 'factor_poblacion']

    df3 = (
        df2
        .merge(clasePoblacion2, on='Codcom', how='left')
        .merge(factor2, on=['Codcom', 'año'], how='left')
    )
    
    verificar_integridad(df3, df2, "Merge Población")

    # Limpieza final
    df3 = df3.drop(columns=['Codcom'])
    
    # Actualizamos df para el resto del proceso
    df = df3

except FileNotFoundError as e:
    print(f"⚠️ Advertencia: No se encontraron archivos de población. {e}")
    df3 = df2 # Continuar sin población

# =========================================
# 13. MAXIMOS HISTORICOS Y ALERTAS (Restored T8-T12)
# =========================================
# --- CORRECCIÓN ROBUSTA PARA SEMANA MÁXIMO HISTÓRICO ---
idx_max_hist = df.groupby(['delito', 'codcom'])['frecuencia'].idxmax()
# Use .loc to extract exact winning rows
info_maximos = df.loc[idx_max_hist, ['delito', 'codcom', 'id_semana', 'semana_detalle']]
info_maximos.rename(columns={
    'id_semana': 'id_semana_max_hist',
    'semana_detalle': 'semana_detalle_max_hist'
}, inplace=True)
df = df.merge(info_maximos, on=['delito', 'codcom'], how='left')

# Alertas
df['alerta_aumento_critico'] = (df['z_score'] > 2) & (df['var_pct_vs_semana_anterior'] > 30)
df['alerta_vs_año_anterior'] = (df['z_score_vs_año_anterior'] > 2) & (df['frecuencia'] > df['max_hist_anual'])

# Casos misma semana año anterior
df_prev_casos = df[['delito', 'codcom', 'año', 'semana_numero', 'frecuencia']].copy()
df_prev_casos['año'] = df_prev_casos['año'] + 1
df_prev_casos = df_prev_casos.rename(columns={'frecuencia': 'casos_misma_semana_año_anterior'})
df_prev_casos = df_prev_casos.drop_duplicates(subset=['delito', 'codcom', 'año', 'semana_numero'])
df = df.merge(df_prev_casos, on=['delito', 'codcom', 'año', 'semana_numero'], how='left')

# Casos Mismo Mes Año Anterior
monthly_cases = df.groupby(['delito', 'codcom', 'año', 'mes'])['frecuencia'].sum().reset_index()
monthly_cases.rename(columns={'frecuencia': 'total_casos_mes_real'}, inplace=True)
prev_year_monthly = monthly_cases.copy()
prev_year_monthly['año'] += 1
prev_year_monthly.rename(columns={'total_casos_mes_real': 'casos_mismo_mes_año_anterior'}, inplace=True)
df = df.merge(prev_year_monthly, on=['delito', 'codcom', 'año', 'mes'], how='left')

# =========================================
# 14. TARJETAS COMPLEX (T19-T25)
# =========================================
# >>>> T19 y T20: Tipologías Críticas <<<<
df_delitos = df[df['delito'] != 'Total']

# 1. Peor Regional por Semana
idx_worst_reg_sem = df_delitos.groupby(['codcom', 'id_semana'])['ranking_comunal_regional'].idxmin()
worst_reg_sem = df_delitos.loc[idx_worst_reg_sem][['codcom', 'id_semana', 'delito', 'ranking_comunal_regional']]
worst_reg_sem.rename(columns={'delito': 't19_delito_sem', 'ranking_comunal_regional': 't19_rank_sem'}, inplace=True)

# 2. Peor Nacional por Semana
df['ranking_nacional_semanal'] = df.groupby(['delito', 'id_semana'])['frecuencia'].rank(method='dense', ascending=False)
idx_worst_nac_sem = df[df['delito'] != 'Total'].groupby(['codcom', 'id_semana'])['ranking_nacional_semanal'].idxmin()
worst_nac_sem = df.loc[idx_worst_nac_sem][['codcom', 'id_semana', 'delito', 'ranking_nacional_semanal']]
worst_nac_sem.rename(columns={'delito': 't20_delito_sem', 'ranking_nacional_semanal': 't20_rank_sem'}, inplace=True)

# Merge
df = df.merge(worst_reg_sem, on=['codcom', 'id_semana'], how='left')
df = df.merge(worst_nac_sem, on=['codcom', 'id_semana'], how='left')

# Shift
df = df.sort_values(['codcom', 'id_semana'])
df['t19_delito_ant'] = df.groupby(['delito', 'codcom'])['t19_delito_sem'].shift(1)
df['t19_rank_ant'] = df.groupby(['delito', 'codcom'])['t19_rank_sem'].shift(1)
df['t20_delito_ant'] = df.groupby(['delito', 'codcom'])['t20_delito_sem'].shift(1)
df['t20_rank_ant'] = df.groupby(['delito', 'codcom'])['t20_rank_sem'].shift(1)

# >>>> T21: Concentración Delictual (Pareto) <<<<
top_delitos = df_delitos.sort_values(['codcom', 'id_semana', 'frecuencia'], ascending=[True, True, False])
top_grp = top_delitos.groupby(['codcom', 'id_semana']).head(3)

def agg_top3(x):
    d = {}
    base_sum = df[(df['codcom']==x.name[0]) & (df['id_semana']==x.name[1]) & (df['delito']=='Total')]['frecuencia'].values
    grand_total = base_sum[0] if len(base_sum)>0 else 1
    i = 1
    for _, row in x.iterrows():
        d[f't21_delito_{i}'] = row['delito']
        d[f't21_val_{i}'] = (row['frecuencia'] / grand_total * 100) if grand_total > 0 else 0
        i += 1
    return pd.Series(d)

top3_info = top_grp.groupby(['codcom', 'id_semana']).apply(agg_top3).reset_index()
df = df.merge(top3_info, on=['codcom', 'id_semana'], how='left')

# >>>> T23: Correlación Corto Plazo <<<<
corr_data = []
comunas_unicas = df['codcom'].unique()
print(f"Iniciando cálculo de correlaciones para {len(comunas_unicas)} comunas...")

for i, c in enumerate(comunas_unicas):
    if (i+1) % 35 == 0: print(f" > Progreso: {i+1}/{len(comunas_unicas)}")
    
    subset = df_delitos[df_delitos['codcom'] == c]
    
    # RANDOM mode logic
    if 'RANDOM' in globals() and RANDOM:
        corr_data.append({'codcom': c, 't23_d1': 'Random_A', 't23_d2': 'Random_B', 't23_val': round(np.random.rand(), 2)})
        continue

    max_sem = subset['id_semana'].max()
    subset_53 = subset[subset['id_semana'] > (max_sem - 53)]
    
    if len(subset_53) < 20:
        corr_data.append({'codcom': c, 't23_d1': 'Insuf. Datos', 't23_d2': '', 't23_val': 0})
        continue

    pivot = subset_53.pivot_table(index='id_semana', columns='delito', values='frecuencia', fill_value=0)
    if pivot.shape[1] < 2:
        corr_data.append({'codcom': c, 't23_d1': 'Mono-delito', 't23_d2': '', 't23_val': 0})
        continue
        
    corr_matrix = pivot.corr().abs()
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    corr_matrix_masked = corr_matrix.mask(mask)
    s = corr_matrix_masked.unstack()
    so = s.sort_values(ascending=False)
    
    try:
        top_pair = so.index[0]
        val = so.iloc[0]
        corr_data.append({'codcom': c, 't23_d1': top_pair[0], 't23_d2': top_pair[1], 't23_val': round(val, 2)})
    except:
        corr_data.append({'codcom': c, 't23_d1': 'Sin Corr', 't23_d2': '', 't23_val': 0})

df_corr = pd.DataFrame(corr_data)
df = df.merge(df_corr, on='codcom', how='left')

# >>>> T25: Aporte Regional Completo (Anterior vs Actual) <<<<
df['aporte_pct_region_ant'] = df.groupby(['delito', 'codcom'])['aporte_pct_region'].shift(1)
df['casos_semana_regional_ant'] = df.groupby(['delito', 'codcom'])['casos_semana_regional'].shift(1)

# Asignar df3 para el bloque siguiente
df3 = df
print("✅ Pipeline ejecutado exitosamente sin errores de memoria ni integridad.")


In [19]:
# =====================================================
# 11. CÁLCULOS PARA TARJETAS BASE (Cards 1-18)
# =====================================================

# --- A. Proyecciones y Tasas ---
df3['semana_numero_safe'] = df3['semana_numero'].replace(0, 1)
df3['proyeccion_anual'] = (df3['acumulado_anual'] / df3['semana_numero_safe']) * 52
df3['tasa_semanal'] = (df3['frecuencia'] / df3['poblacion']) * 100000
df3['tasa_proyectada_anual'] = (df3['proyeccion_anual'] / df3['poblacion']) * 100000

# --- B. Agregaciones Nacionales ---
grp_nac = df3.groupby(['delito', 'id_semana'])
df3['tasa_proyectada_nacional'] = grp_nac['proyeccion_anual'].transform('sum') / grp_nac['poblacion'].transform('sum') * 100000
df3['tasa_semanal_nacional'] = grp_nac['frecuencia'].transform('sum') / grp_nac['poblacion'].transform('sum') * 100000

# --- C. Agregaciones Regionales ---
grp_reg = df3.groupby(['Codreg', 'delito', 'id_semana'])
df3['tasa_proyectada_regional'] = grp_reg['proyeccion_anual'].transform('sum') / grp_reg['poblacion'].transform('sum') * 100000
df3['tasa_semanal_regional'] = grp_reg['frecuencia'].transform('sum') / grp_reg['poblacion'].transform('sum') * 100000
df3['casos_semana_regional'] = grp_reg['frecuencia'].transform('sum')
df3['aporte_pct_region'] = (df3['frecuencia'] / df3['casos_semana_regional'].replace(0, np.nan)) * 100

# --- D. Rankings ---
df3['ranking_regional_proy_anual'] = df3.groupby(['Codreg', 'delito', 'id_semana'])['proyeccion_anual'].rank(method='dense', ascending=False)
df3['ranking_nacional_semanal'] = df3.groupby(['delito', 'id_semana'])['frecuencia'].rank(method='dense', ascending=False)
df3['ranking_nacional_proy_anual'] = df3.groupby(['delito', 'id_semana'])['proyeccion_anual'].rank(method='dense', ascending=False)

grp_cluster = df3.groupby(['clase_poblacion', 'delito', 'id_semana'])
df3['ranking_cluster_semanal'] = grp_cluster['frecuencia'].rank(method='dense', ascending=False)
df3['ranking_cluster_proy_anual'] = grp_cluster['proyeccion_anual'].rank(method='dense', ascending=False)

# Shifts de Rankings
df3 = df3.sort_values(['Codreg', 'delito', 'codcom', 'id_semana'])
g_temp = df3.groupby(['delito', 'codcom'])
df3['ranking_regional_proy_anual_anterior'] = g_temp['ranking_regional_proy_anual'].shift(1)
df3['ranking_nacional_semanal_anterior'] = g_temp['ranking_nacional_semanal'].shift(1)
df3['ranking_nacional_proy_anual_anterior'] = g_temp['ranking_nacional_proy_anual'].shift(1)
df3['ranking_cluster_semanal_anterior'] = g_temp['ranking_cluster_semanal'].shift(1)

# --- E. Stats Adicionales ---
df3['proyeccion_mes_actual'] = df3['media_movil_4s'] * 4.33
df3['promedio_diario_semanal'] = df3['frecuencia'] / 7
df3['promedio_diario_historico'] = df3['promedio_hist'] / 7

total_semanal_comuna = df3.groupby(['codcom', 'id_semana'])['frecuencia'].transform('sum')
df3['share_delito_semanal'] = (df3['frecuencia'] / total_semanal_comuna.replace(0, np.nan)) * 100

df3.drop(columns=['semana_numero_safe'], inplace=True, errors='ignore')

print("DataFrame Final Listo. Columnas:")
print(df3.columns.tolist())
print(f"\nTotal columnas: {len(df3.columns)}")

In [20]:
# =========================================
# 12. VALIDACIÓN DETALLADA (SIMULACIÓN DASHBOARD)
# =========================================

santiago = df3[(df3['codcom'] == 13101) & (df3['delito'] == 'Total')].sort_values('id_semana')
ultima = santiago.iloc[-1]

print(f"\n=== VALIDACIÓN SANTIAGO {ultima['semana_detalle']} ===\n")
print(f"[T1] Casos Actuales: {ultima['casos_semana_actual']}")
print(f"[T2] Casos Anterior: {ultima['casos_semana_anterior']} (Delta: {ultima['delta']})")
print(f"[T3] Acumulado Anual: {ultima['acumulado_anual']}")
print(f"[T4] Media Movil 4S: {ultima['media_movil_4s']:.1f}")
print(f"[T5] Promedio Histórico: {ultima['promedio_hist']:.1f}")
print(f"[T6] Z-Score: {ultima['z_score']:.2f} ({ultima['conclusion_z']})")
print(f"[T7] Racha: {ultima['racha']} semanas")
print(f"[T8] Max Historico Semana: {ultima['semana_detalle_max_hist']}")
print(f"[T9] Alerta Aumento Critico: {ultima['alerta_aumento_critico']}")
print(f"[T10] Alerta Año Anterior: {ultima['alerta_vs_año_anterior']}")
print(f"[T11] Casos Misma Sem Año Ant: {ultima['casos_misma_semana_año_anterior']}")
print(f"[T12] Casos Mismo Mes Año Ant: {ultima['casos_mismo_mes_año_anterior']}")
print(f"[T13] Ranking Reg Semanal: {ultima['ranking_comunal_regional']}")
print(f"[T14] Ranking Nac Semanal: {ultima['ranking_nacional_semanal']}")
print(f"[T15] Ranking Cluster Semanal: {ultima['ranking_cluster_semanal']}")
print(f"[T16] Proyección Anual: {ultima['proyeccion_anual']:.0f}")
print(f"[T17] Tasa Semanal: {ultima['tasa_semanal']:.1f}")
print(f"[T18] Tasa Proyectada: {ultima['tasa_proyectada_anual']:.1f}")
print(f"[T19] Peor Ranking Regional Actual: {ultima['t19_delito_sem']} (Pos {ultima['t19_rank_sem']:.0f})")
print(f"[T20] Peor Ranking Nacional Actual: {ultima['t20_delito_sem']} (Pos {ultima['t20_rank_sem']:.0f})")
print(f"[T21] Top 1 Delito: {ultima['t21_delito_1']} ({ultima['t21_val_1']:.1f}%)")
print(f"[T23] Correlación Fuerte (53 Sem): {ultima['t23_d1']} vs {ultima['t23_d2']} ({ultima['t23_val']:.2f})")
print(f"[T25] Aporte Regional: {ultima['aporte_pct_region']:.1f}% (Ant: {ultima['aporte_pct_region_ant']:.1f}%)")


In [ ]:
df3.columns

In [ ]:
# =========================================
# GUARDADO POR COMUNA (data/stop/{codcom})
# =========================================
import os
output_dir = r'D:\GitHub\STOP_WEB3\web_js\data\stop'
os.makedirs(output_dir, exist_ok=True)

for i in df3["codcom"].unique():
    aux = df3[df3["codcom"] == i]
    aux.to_json(fr'{output_dir}/{i}', orient='records', compression='gzip', indent=None)

print(f"Guardados {df3['codcom'].nunique()} archivos por comuna.")